In [1]:
!pip install pandas
!pip install selenium
!pip install 2captcha-python


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
from time import sleep
from datetime import datetime

import pandas as pd

from twocaptcha import TwoCaptcha

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

URL_PROFESSORES_SIGAA = "https://sigaa.ufpi.br/sigaa/public/departamento/professores.jsf?id=144"
CAPTCHA2_API_KEY = "b53520e9ff8d77c6751932a1a9133abb"
LATES_SITE_KEY = "6LeDv6QUAAAAAP_ZK5AXrsNRQjxfbjgZLV5_YHyy"

In [3]:
def resolver_captcha(site_key, page_url):
    api_key = CAPTCHA2_API_KEY

    solver = TwoCaptcha(api_key)

    try:
        tempo_inicial = datetime.now()

        result = solver.recaptcha(
            sitekey=site_key,
            url=page_url,
            # invisible=1,
            # enterprise=1
        )

        print(result["code"][:100])

        return result["code"]
    except Exception as e:
        print(e)
    finally:
        tempo_final = datetime.now()

        print(f"Tempo de resolução do captcha: {tempo_final - tempo_inicial}")

In [4]:
def coletar_dados_lattes(driver, wait, dados_docente):
    try:    
        driver.get(dados_docente["Lattes"])

        url_atual = driver.current_url

        codigo_captcha = resolver_captcha(LATES_SITE_KEY, url_atual)

        driver.execute_script("""
            document.getElementById('tokenCaptchar').value = arguments[0];
        """, codigo_captcha)

        sleep(2)

        driver.execute_script("""
            document.getElementById('formulario').submit();
        """)

        sleep(2)

        categorias = driver.find_elements(By.CLASS_NAME, "title-wrapper")
        producoes_docente = []
        bancas_docente = []

        for categoria in categorias:
            titulo_categoria = categoria.text

            if "Produções" in titulo_categoria:
                # print(categoria.text, "\n")
                producoes = categoria.find_elements(By.CLASS_NAME, "layout-cell.layout-cell-11")
                for producao in producoes:
                    doi = producao.find_elements(By.CLASS_NAME, "icone-doi")
                    doi_url = doi[0].get_attribute("href") if doi else ""

                    producoes_docente.append({
                        "Título": producao.text.strip(),
                        "URL": doi_url
                    })
            elif "Bancas" in titulo_categoria:
                # print(categoria.text, "\n")
                bancas = categoria.find_elements(By.CLASS_NAME, "layout-cell.layout-cell-11")
                for banca in bancas:
                    doi = banca.find_elements(By.CLASS_NAME, "icone-doi")
                    doi_url = doi[0].get_attribute("href") if doi else ""

                    bancas_docente.append({
                        "Título": banca.text.strip(),
                        "URL": doi_url
                    })

        dados_docente["Produções"] = producoes_docente
        dados_docente["Bancas"] = bancas_docente
    except Exception as e:
        print(f"Falha ao coletar dados do Lattes: {e}")

    return dados_docente

In [5]:
def coletar_dados_sigaa(driver, wait):
    try:
        dados_docentes = []

        driver.get(URL_PROFESSORES_SIGAA)

        tabelas = wait.until(
            EC.presence_of_all_elements_located(
                (By.TAG_NAME, "tbody")
            )
        )

        for tabela in tabelas:
            for linha in tabela.find_elements(By.TAG_NAME, "tr"):
                colunas = linha.find_elements(By.TAG_NAME, "td")

                url_lattes = colunas[3].find_elements(By.LINK_TEXT, "Currículo Lattes")
                url_lattes = url_lattes[0].get_attribute("href") if url_lattes else ""

                dados_docentes.append({
                    "Foto": colunas[0].find_element(By.TAG_NAME, "img").get_attribute("src"),
                    "Nome": colunas[1].text.strip(),
                    "Formação": colunas[2].get_attribute("textContent").strip(),
                    "Lattes": url_lattes
                })
    except Exception as e:
        print(f"Falha ao coletar dados do SIGAA: {e}")

    return dados_docentes

In [6]:
def main():
    driver = webdriver.Chrome()

    wait = WebDriverWait(driver, timeout=30)

    dados_docentes = coletar_dados_sigaa(driver, wait)

    for indice_docente, dados_docente in enumerate(dados_docentes):
        print(f"Executando para o prestador {dados_docente['Nome']}: {indice_docente + 1} de {len(dados_docentes)}")

        dados_docente = coletar_dados_lattes(driver, wait, dados_docente)

        break

    driver.quit()

    return dados_docentes

In [7]:
dados_docentes = main()

Executando para o prestador ANDRE CASTELO BRANCO SOARES (DOUTOR): 1 de 25
0cAFcWeA6P_DgHGuih1xde5RTeTdWyHGpG7OpVOwPgpeJaU1hvWSjynnou2SQI4wgBaJb8_DHnIAD1W3296TxbB1Xm5j93Wz56RO
Tempo de resolução do captcha: 0:00:57.315176


In [8]:
dados_docentes[0]

{'Foto': 'https://sigaa.ufpi.br/sigaa/verFoto?idFoto=13151&key=5dc20fe47a0c78d98cb2c9c06c0a1ec0',
 'Nome': 'ANDRE CASTELO BRANCO SOARES (DOUTOR)',
 'Formação': 'Doutor em Ciência da Computação (CIn/UFPE 2009)\nMestre em Redes de Computadores (Universidade Salvador 2004)\nBarachel em Ciência da Computação ...',
 'Lattes': 'http://lattes.cnpq.br/4545154317245176',
 'Produções': [{'Título': 'MELO, CARLOS ; MIQUEIAS, JOSÉ ; MESSIAS, JOHNNATAN ; GONÇALVES, GLAUBER ; SILVA, FRANCISCO AIRTON ; SOARES, ANDRÉ ; ARAUJO, JEAN . A stochastic performance model for evaluating ethereum layer-2 rollups. Future Generation Computer Systems, v. 179, p. 108316, 2026. Citações:1',
   'URL': 'https://dx.doi.org/10.1016/j.future.2025.108316'},
  {'Título': 'LACERDA, JURANDIR C. ; SOUSA, CARLOS E.B. ; MORAIS, ALINE G. ; CARTAXO, ADOLFO V.T. ; SOARES, ANDRÉ C.B. . Machine learning-based algorithm for core allocation in spatial division multiplexing elastic optical networks. OPTICAL FIBER TECHNOLOGY, v. 91, p. 

In [13]:
elementos_curriculos = wait.until(
    EC.presence_of_all_elements_located(
        (By.LINK_TEXT, "Currículo Lattes")
    )
)

links_curriculos = []

for elemento_curriculo in elementos_curriculos:
    link_curriculo = elemento_curriculo.get_attribute("href")
    links_curriculos.append(link_curriculo)

In [ ]:
tabelas = wait.until(
    EC.presence_of_all_elements_located(
        (By.TAG_NAME, "tbody")
    )
)

dados_docentes = []

for tabela in tabelas:
    for linha in tabela.find_elements(By.TAG_NAME, "tr"):
        colunas = linha.find_elements(By.TAG_NAME, "td")

        url_lattes = colunas[3].find_elements(By.LINK_TEXT, "Currículo Lattes")
        url_lattes = url_lattes[0].get_attribute("href") if url_lattes else ""

        dados_docentes.append({
            "Foto": colunas[0].find_element(By.TAG_NAME, "img").get_attribute("src"),
            "Nome": colunas[1].text.strip(),
            "Formação": colunas[2].get_attribute("textContent").strip(),
            "Lattes": url_lattes
        })

[{'Foto': 'https://sigaa.ufpi.br/sigaa/verFoto?idFoto=13151&key=5dc20fe47a0c78d98cb2c9c06c0a1ec0',
  'Nome': 'ANDRE CASTELO BRANCO SOARES (DOUTOR)',
  'Formação': 'Doutor em Ciência da Computação (CIn/UFPE 2009)\nMestre em Redes de Computadores (Universidade Salvador 2004)\nBarachel em Ciência da Computação ...',
  'Lattes': 'http://lattes.cnpq.br/4545154317245176'},
 {'Foto': 'https://sigaa.ufpi.br/sigaa/img/no_picture.png',
  'Nome': 'ANTONIO COSTA DE OLIVEIRA (DOUTOR)',
  'Formação': 'Doutorado em Engenharia de Produção (2002) - Escola Politécnica - USP\nMestrado em Otimização e Pesquisa Operacional (1987) - UNICAMP\nGraduação em ...',
  'Lattes': 'http://lattes.cnpq.br/6511083001417681'},
 {'Foto': 'https://sigaa.ufpi.br/sigaa/verFoto?idFoto=3634955&key=38c142949c19417a17277639a6b1ea67',
  'Nome': 'ANTONIO HELSON MINEIRO SOARES (DOUTOR)',
  'Formação': 'Possui graduação em Bacharelado em Ciência da Computação pela Universidade Federal do Piauí (UFPI) em 2006.2, obteve o título de M

In [ ]:
for indice_docente, dados_docente in enumerate(dados_docentes):
    try:
        driver.get(dados_docente["Lattes"])

        current_url = driver.current_url

        codigo_captcha = resolver_captcha(LATES_SITE_KEY, current_url)

        driver.execute_script("""
            document.getElementById('tokenCaptchar').value = arguments[0];
        """, codigo_captcha)

        sleep(2)

        driver.execute_script("""
            document.getElementById('formulario').submit();
        """)

        sleep(2)

        categorias = driver.find_elements(By.CLASS_NAME, "title-wrapper")
        producoes_docente = []
        bancas_docente = []

        for categoria in categorias:
            titulo_categoria = categoria.text

            if "Produções" in titulo_categoria:
                # print(categoria.text, "\n")
                producoes = categoria.find_elements(By.CLASS_NAME, "layout-cell.layout-cell-11")
                for producao in producoes:
                    producoes_docente.append(producao.text.strip())
            elif "Bancas" in titulo_categoria:
                # print(categoria.text, "\n")
                bancas = categoria.find_elements(By.CLASS_NAME, "layout-cell.layout-cell-11")
                for banca in bancas:
                    bancas_docente.append(banca.text.strip())
    except Exception as e:
        print(f"Falha ao coletar dados do Lattes: {e}")

0cAFcWeA6XSKZKlQHZYMquptNAxIAUgsgZesq5fpRRHH85Y6ErTPzAt2CZUV9Dd4NQK9B2LEwkfAPpnNnC2qo8xDhTuCYhNrdEyl
0cAFcWeA6emIqoZ6t-1tPZwTiYc4d3I50zY8JxDJEbOTQhYEXyxZm_rm7nhBr9n1GaM6MQMMkD5QBMQJQW5k4_DIf4UruzuDOzN5
0cAFcWeA7iN2SI6xts-Y5XqyE182QDWn_P21P6lsYVMB8oupptw6Au0NVZNMzPSESeramYOzSKP2ntpZBZ2Xz5TZvHORqS4aVJIv


In [ ]:
categorias = driver.find_elements(By.CLASS_NAME, "title-wrapper")
producoes_docente = []
bancas_docente = []

for categoria in categorias:
    titulo_categoria = categoria.text

    if "Produções" in titulo_categoria:
        # print(categoria.text, "\n")
        producoes = categoria.find_elements(By.CLASS_NAME, "layout-cell.layout-cell-11")
        for producao in producoes:
            producoes_docente.append(producao.text.strip())
    elif "Bancas" in titulo_categoria:
        # print(categoria.text, "\n")
        bancas = categoria.find_elements(By.CLASS_NAME, "layout-cell.layout-cell-11")
        for banca in bancas:
            bancas_docente.append(banca.text.strip())

SOARES, ANTONIO; RÂBELO, RICARDO ; DELBEM, ALEXANDRE . Optimization based on Phylogram Analysis. Expert Systems with Applications, v. 78, p. 32-50, 2017. Citações:40|43
Soares, Antônio; Santos, Valéria ; Toledo, Cláudio ; Osório, Fernando ; DELBEM, ALEXANDRE . ND-NCD: Environmental Characteristics Recognition and Novelty Detection for Mobile Robots Control and Navigation. Communications in Computer and Information Science. 1ed.: Springer International Publishing, 2016, v. 619, p. 192-209.
SOARES, A. H. M.; Delbem, A. C. B. . Detecção de Correlação em Dados Complexos usando NCD. ERIPI. 1ed.: , 2015, v. , p. 1-.
DE ARAUJO, FRANCISCO N.C. ; MACHADO, VINICIUS P. ; SOARES, ANTONIO H.M. ; DE M.S. VERAS, RODRIGO . Automatic Cluster Labeling Based on Phylogram Analysis. In: 2018 International Joint Conference on Neural Networks (IJCNN), 2018, Rio de Janeiro. 2018 International Joint Conference on Neural Networks (IJCNN), 2018. p. 1.
ARAUJO, F. N. C. ; SOARES, A. H. M. ; MACHADO, V. P. ; Rabelo

In [ ]:
for dado_professor in dados_professores:
    driver.get(dado_professor["Lattes"])

    # iframe = wait.until(
    #     EC.presence_of_element_located(
    #         (By.TAG_NAME, "iframe")
    #     )
    # )

    # driver.switch_to.frame(iframe)

    # checkbox_captcha = wait.until(
    #     EC.presence_of_element_located(
    #         (By.CLASS_NAME, "rc-anchor-checkbox-holder")
    #     )
    # )

    # checkbox_captcha.click()

    codigo_captcha = result["code"]

    print(codigo_captcha)

    # driver.execute_script(
    #     f"document.getElementById('g-recaptcha-response').innerHTML = '{codigo_captcha}'"
    # )

    # driver.execute_script(
    #     "document.getElementById('submitBtn').removeAttribute('disabled');"
    # )

    # driver.find_element(By.ID, "submitBtn").click()

    sleep(2)

    # botao_submeter = driver.find_element(By.ID, "submitBtn")
    # driver.execute_script("document.getElementById('submitBtn').click();")

    # formulario = driver.find_element(By.NAME, "visualizacvForm")
    # formulario.submit()

    driver.execute_script("""
        onloadCallback();
    """)
    
    driver.execute_script(
        f"document.getElementById('tokenCaptchar').value = '{codigo_captcha}';"
    )

    # driver.execute_script("document.getElementById('submitBtn').click();")

    break


0cAFcWeA521YVrylK22Q808ERCsX6-BjiaUv7zSY4dYzaSZpCMuQ1YDLMqr_CwBXt_tMaKj97DknjzFKIczdL1NlXOptivOvkBAG9Bcwsi0k0pXiOCTef5ayIV6w0mTx00icOIL5UhLIUZ2-AZHkB4rZ1bEg3y8DdCGYHkWBSOPbzD0YMlhF35mZKvLECNXaMerzeWY6u5DrTpv-Wl6Rl-YCfAzB8Ec5kPEmL5f9mmBPeRXnKn3OElorru_vcfXMXR0w9RDLqDJ9rADb83oyntpQWysZTQybIwCxfL3FhEoGbddBs5oXm1sPWK4Wf0yLf7AylDyIQNyidDCsnkou203zXLdnEUHjdMHV3i4lbjO9H1yupEa6hcOdb0p9rmXkcCvEhcrCNOSCbXJj_QvixsE0gs4002Yp1-9HplSroRCeO0PxDcNTauu2UzLDUcxggZiyZql24LZzIAxBm4DAD4zXl6rVNRhM6PupzNNMP4BzOlkHS5V5jKsZZRk3RwOR4_RWnm_SeIgMxfoo2YVfIjbKtfT6OeqEnt4i4HpuIyeMnZ_Qel8mCGF0L3ZR8ELqryVEKwRfMfYxaLuTLHVx8evsLC6AQYQEwF9cdM4syE-4W_5ZCxfYzOFTi0um_6VDQ2X9Uxi8eSc4VLocZoSSDlmr8Nh0S0Xj9ika-V_wfs24mMtOf2PttWBgqopEsHU7qhzysh76m9ZvRGOq98T_W6laoGDaoL3rXyQ98lVEaO8oDzpBv-tbTDTpLAfYwzpZKNdW1VOUWuhsRd-ZeO4XpNeHz8m-ZubABIE3f6MiNiFjp2uXV1WajsYe6_sEbAz01F709ilaVRDDJDEpadmvo_YlgaJX0hjsLBsrDdDR9dZzjqN-M22KF5wLlYLF4Nhrvv5Ajcff_aH2_ekgdKwGotIVjV2Apz-l3mPWKwsdsA9lol6g9wbXpCcdu7SScOgN3LfgsPO4p7MbGa5nZ2yDlmFeJcrbursbmxPot1Zm

In [87]:
import os

from twocaptcha import TwoCaptcha

api_key = "b53520e9ff8d77c6751932a1a9133abb"

solver = TwoCaptcha(api_key)

try:
    result2 = solver.recaptcha(
        sitekey="6LeDv6QUAAAAAP_ZK5AXrsNRQjxfbjgZLV5_YHyy",
        url="https://buscatextual.cnpq.br/buscatextual/visualizacv.do?metodo=apresentar&id=K4773498D2",
        enterprise=1
    )
except Exception as e:
    print(e)

else:
    print(f"result: {str(result2)}")

result: {'captchaId': '82196735183', 'code': '0cAFcWeA5Qi1PQ80tM954CVK3LrjnjH5ZKh2WJQM1LArYjMnMfnlNdlZQwtfI3uTjTia3JgMzplB8i5gWefuPed1FGGwS8EUxEcdDDNlAHVQGvTK5278rSTUrfLfzgnL_Zd2JlULFgx-gwqxrDtSNTM6ma6olFXWM2EU23SP8xAIDLMwjmF-CFBhdkxRNgUdV-4kh_FEWdM_ylS1eKzflJhldve0kJyiGDL4sPVTG7GiQgT0mYBMwdOowSh0bHmjGuE-vDe5L9iJ7TMmumxipjbd0XItXBpMD9dgyQvXxbTHw5ZhoqWMp2a7Q1jAJpdFlrfKL84Xy13m5F3vyGTou5gu_UoBtCbESFK_WgfINZBsOyYzLg0oMP75SIzgrK9jBWqXQPU-c_ZzRYVdW_em5d1qmUYwkRa7nb85dcf3CmtCiRG6bslgFqofwUjIP0kgNiLqF4uJFAW6P0d_9qQfhyXljRDZRDvCsW-yChUDtNlUvyIVPuKI4zzY89GXH4J9iWZ4MvvibXhIEilhRtUYJOfP4ANcAVHsKs9kYJ8h8f5JjiqnJfFlFQV-d3lDBHZ8QdE-UBFmEKp2Ikm-CUt2xfWuTRvBIXMyZhKYmLl17dsYY0hywiFeoR262gwxb1VdbEzdnwavyXBJ0HVZIrVFAAg4648ZQkCpTBGufQay6l2HDBhnnNLqO3-47vTBLsLSUsz14ItKmuqHOTRYWvJCzlVwF65o3_laMNH3mp2ErTbUb5frQ6W-OGForVrAXIGA-LCvZZQQ5Xt9arLFS_NkLUJK_rSMiubDXnhUJwNhOcJ94FtHNB6l3obkmaUW3X8-63RArau9AnpBSqIOfp4z0b7_7UaGY7TNCzw3rwyRG7R_Mnkz-j5CtedSAolmurOHd5bnKoeW_-ayxsPL_ge_Z4lm7xpjpI8HpClVRT8j4VDKsyzh8AjqNxzN8a

In [ ]:
for iframe in iframes:
    print(iframe.get_attribute("innerHTML"))

In [91]:
driver.execute_script("""
    verifyCallback(arguments[0]);
""", codigo_captcha)

In [ ]:
for dado_professor in dados_professores:
    driver.get(dado_professor["Lattes"])

    # iframes = driver.find_elements(By.TAG_NAME, "iframAe")

    # iframes = wait.until(
    #     EC.presence_of_all_elements_located(
    #         (By.TAG_NAME, "iframe")
    #     )
    # )

    # print(f"Quantidade de iframes: {len(iframes)}")

    # driver.switch_to.frame(iframes[0])

    iframe = wait.until(
        EC.presence_of_element_located(
            (By.TAG_NAME, "iframe")
        )
    )

    driver.switch_to.frame(iframe)

    # checkbox_captcha = wait.until(
    #     EC.presence_of_element_located(
    #         (By.CLASS_NAME, "rc-anchor-checkbox-holder")
    #     )
    # )

    # checkbox_captcha.click()

    # resposta_captcha = wait.until(
    #     EC.presence_of_element_located(
    #         (By.ID, "g-recaptcha-response")
    #     )
    # )

    codigo_captcha = result["code"]

    # print(codigo_captcha)

    driver.execute_script(
        "document.getElementById('g-recaptcha-response').innerHTML = " + "'" + codigo_captcha + "'"
    )

    # resposta_captcha = driver.find_element(By.ID, "g-recaptcha-response")

    # print(resposta_captcha.get_attribute("outterHTML"))

    # driver.execute_script("arguments[0].value=arguments[1]", resposta_captcha, codigo_captcha)

    # driver.find_element(By.ID, "submitBtn").click()

    break
# document.getElementsByClassName("rc-anchor-checkbox-holder")
